# 03_train_slovenia_model

Trening prvega explainable modela za slovenski nepremičninski trg na ETN datasetu.

## Cilji
- naloži `slovenia_apartments_modeling_v1.parquet` iz prejšnjega notebooka
- naredi **časovni split**
  - **train**: do konca 2023
  - **validation**: leto 2024
  - **test**: leto 2025
- primerja 2 modela:
  - **Model A**: brez `trznost_group`
  - **Model B**: z `trznost_group`
- target:
  - **`log_price_per_m2`**
- model:
  - **RandomForestRegressor**
- pripravi:
  - metrike
  - feature importance
  - test napovedi
  - LIME razlage
  - shranjen pipeline za Streamlit

## Zakaj tak split
- ne mešamo prihodnosti v preteklost
- 2025 ostane pravi holdout za evalvacijo
- `sale_date` / `sale_year` uporabljamo kot časovno resnico, ne `SOURCE_YEAR`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls /content

drive  sample_data


In [ ]:
!ls "/content/gdrive/MyDrive/etn_2021_2025/processed_outputs"


ls: cannot access '/content/gdrive/MyDrive/etn_2021_2025/processed_outputs': No such file or directory


In [ ]:
!ls "/content/drive/MyDrive/etn_2021_2025/processed_outputs"

model_outputs
slovenia_apartments_modeling_v1.csv
slovenia_apartments_modeling_v1_feature_columns.json
slovenia_apartments_modeling_v1.parquet
slovenia_apartments_modeling_v1_summary.json


In [ ]:
# Če kaj manjka, odkomentiraj:
# !pip install pandas pyarrow scikit-learn joblib lime

import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from sklearn.metrics import root_mean_squared_error
    def rmse(y_true, y_pred):
        return float(root_mean_squared_error(y_true, y_pred))
except ImportError:
    def rmse(y_true, y_pred):
        return float(np.sqrt(mean_squared_error(y_true, y_pred)))
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
import joblib

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

SEED = 42
np.random.seed(SEED)
random.seed(SEED)


## 1) LIME import

Če `lime` ni nameščen, ga namesti.


In [ ]:
try:
    from lime.lime_tabular import LimeTabularExplainer
    print("LIME import OK")
except ModuleNotFoundError:
    import sys
    !{sys.executable} -m pip install -q lime
    from lime.lime_tabular import LimeTabularExplainer
    print("LIME installed")


LIME import OK


## 2) Nastavitve poti

Ta verzija uporablja **direktno pot do dataseta**, da se izognemo težavam z auto-discovery.

Če uporabljaš Colab + Google Drive, je najbolj varno:
1. najprej mountaš Drive,
2. nato nastaviš **točno pot do parquet/csv** datoteke.


In [ ]:
# Primeri:
# data_path = Path("/content/gdrive/MyDrive/etn_2021_2025/processed_outputs/slovenia_apartments_modeling_v1.parquet")
# data_path = Path("/content/drive/MyDrive/etn_2021_2025/processed_outputs/slovenia_apartments_modeling_v1.parquet")
# data_path = Path("/content/processed_outputs/slovenia_apartments_modeling_v1.parquet")
# data_path = Path("/mnt/data/slovenia_apartments_modeling_v1.parquet")

data_path = Path("/content/drive/MyDrive/etn_2021_2025/processed_outputs/slovenia_apartments_modeling_v1.parquet")
print("data_path =", data_path)
print("exists    =", data_path.exists())

if not data_path.exists():
    alt_csv = data_path.with_suffix(".csv")
    print("alt_csv   =", alt_csv)
    print("csv exists=", alt_csv.exists())
    if alt_csv.exists():
        data_path = alt_csv
    else:
        parent = data_path.parent
        print("\nKaj Python vidi v parent mapi:")
        if parent.exists():
            for p in sorted(parent.iterdir()):
                print(p.name)
        raise FileNotFoundError(f"Datoteka ne obstaja: {data_path}")

DATA_DIR = data_path.parent
OUT_DIR = DATA_DIR / "model_outputs"
print("\nDATA_DIR =", DATA_DIR)
print("OUT_DIR  =", OUT_DIR)


data_path = /content/drive/MyDrive/etn_2021_2025/processed_outputs/slovenia_apartments_modeling_v1.parquet
exists    = True

DATA_DIR = /content/drive/MyDrive/etn_2021_2025/processed_outputs
OUT_DIR  = /content/drive/MyDrive/etn_2021_2025/processed_outputs/model_outputs


## 3) Potrdi modeling dataset

V tej verziji dataset **ni več iskan samodejno**.  
Uporabi se `data_path`, ki si ga nastavil zgoraj.


In [ ]:
print("Using:", data_path)
print("suffix:", data_path.suffix)
print("exists:", data_path.exists())

# OUT_DIR ustvarimo šele zdaj, ko vemo, da je data_path pravilen
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR exists:", OUT_DIR.exists())


Using: /content/drive/MyDrive/etn_2021_2025/processed_outputs/slovenia_apartments_modeling_v1.parquet
suffix: .parquet
exists: True
OUT_DIR exists: True


## 4) Naloži podatke

In [ ]:
if data_path.suffix == ".parquet":
    df = pd.read_parquet(data_path)
else:
    df = pd.read_csv(data_path, low_memory=False)

if "sale_date" in df.columns:
    df["sale_date"] = pd.to_datetime(df["sale_date"], errors="coerce")

print("shape:", df.shape)
display(df.head(3))
display(df.dtypes.head(20).to_frame("dtype"))


shape: (25529, 42)


,ID_POSLA,sale_date,sale_year,sale_month,sale_quarter,SOURCE_YEAR,price_eur,price_per_m2,log_price,log_price_per_m2,municipality,settlement,ko_name,SIFRA_KO,e_centroid,n_centroid,VRSTA_DELA_STAVBE,VRSTA_DELA_STAVBE_OPIS,build_year,property_age,area_m2,area_source,usable_area_m2,building_area_m2,rooms,parking_spaces,has_atrium,LEGA_DELA_STAVBE_V_STAVBI,NADSTROPJE_DELA_STAVBE,NOVOGRADNJA,STAVBA_JE_DOKONCANA,GRADBENA_FAZA,VKLJUCENOST_DDV,TRZNOST_POSLA,TRZNOST_POSLA_OPIS,trznost_group,VRSTA_KUPOPRODAJNEGA_POSLA,VRSTA_KUPOPRODAJNEGA_POSLA_OPIS,VRSTA_AKTA,VRSTA_AKTA_OPIS,n_deli,n_parcel
0,525820,2020-12-10,2020,12,4,2021,35000.0,1166.666667,10.463103,7.061906,KAMNIK,KAMNIK,PODGORJE,1908,469375.55,118784.83,2,Stanovanje,1965,55,30.0,PRODANA_POVRSINA,25.5,29.9,NaN,NaN,NaN,nadstropje,NaN,NaN,1,NaN,0,2.0,Tržen posel – neustrezni podatki,market_inadequate,1,Prodaja nepremičnin na prostem trgu,1,Osnovna pogodba,1,0
1,532981,2021-02-24,2021,2,1,2021,73000.0,1280.701754,11.198215,7.155163,MARIBOR,MARIBOR,MARIBOR GRAD,657,550055.31,157775.43,2,Stanovanje,1950,71,57.0,PRODANA_POVRSINA,45.9,56.7,NaN,NaN,NaN,nadstropje,NaN,NaN,1,NaN,0,2.0,Tržen posel – neustrezni podatki,market_inadequate,1,Prodaja nepremičnin na prostem trgu,1,Osnovna pogodba,1,0
2,555215,2021-08-13,2021,8,3,2021,117000.0,1800.000000,11.669929,7.495542,MARIBOR,MARIBOR,TABOR,659,549707.36,156744.89,2,Stanovanje,1971,50,65.0,PRODANA_POVRSINA,57.8,64.8,NaN,NaN,NaN,nadstropje,NaN,NaN,1,NaN,0,2.0,Tržen posel – neustrezni podatki,market_inadequate,1,Prodaja nepremičnin na prostem trgu,1,Osnovna pogodba,1,0


,dtype
ID_POSLA,int64
sale_date,datetime64[ns]
sale_year,int32
sale_month,int32
sale_quarter,int32
SOURCE_YEAR,int64
price_eur,float64
price_per_m2,float64
log_price,float64
log_price_per_m2,float64


## 5) Osnovni sanity checks

In [ ]:
required_cols = ["sale_date", "log_price_per_m2", "price_per_m2", "price_eur", "area_m2", "sale_year"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Manjkajo obvezni stolpci: {missing_required}")

print("sale_date min:", df["sale_date"].min())
print("sale_date max:", df["sale_date"].max())

display((df.isna().mean() * 100).sort_values(ascending=False).head(25).round(2).to_frame("missing_%"))


sale_date min: 2007-06-19 00:00:00
sale_date max: 2025-12-30 00:00:00


,missing_%
rooms,100.00
parking_spaces,100.00
has_atrium,100.00
NADSTROPJE_DELA_STAVBE,100.00
GRADBENA_FAZA,99.70
NOVOGRADNJA,95.18
LEGA_DELA_STAVBE_V_STAVBI,0.76
settlement,0.45
usable_area_m2,0.18
building_area_m2,0.16


## 6) Konfiguracija featurejev

Tukaj namerno izpustimo featureje, ki so skoraj prazni ali uporabniško težko razložljivi.

### V1 featureji
**Numerični**
- `area_m2`
- `property_age`
- `build_year`
- `sale_year`
- `sale_month`
- `e_centroid`
- `n_centroid`
- `usable_area_m2`
- `building_area_m2`

**Kategorični**
- `municipality`
- `settlement`
- `ko_name`
- `VKLJUCENOST_DDV`
- `LEGA_DELA_STAVBE_V_STAVBI`
- `NOVOGRADNJA`
- `STAVBA_JE_DOKONCANA`
- `GRADBENA_FAZA`

**Eksperimentalni**
- `trznost_group`


In [ ]:
print("Using:", data_path)
print("suffix:", data_path.suffix)
print("exists:", data_path.exists())

# OUT_DIR ustvarimo šele zdaj, ko vemo, da je data_path pravilen
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("OUT_DIR exists:", OUT_DIR.exists())


Using: /content/drive/MyDrive/etn_2021_2025/processed_outputs/slovenia_apartments_modeling_v1.parquet
suffix: .parquet
exists: True
OUT_DIR exists: True


## 7) Pripravi data frame za model

- odstrani vrstice brez targeta
- kategorialne featureje pretvori v **object/string placeholder**, ne v pandas `StringDtype`
- to je pomembno zaradi kompatibilnosti s starejšimi verzijami `scikit-learn`, kjer lahko `pd.NA` povzroči napako:
  - `TypeError: boolean value of NA is ambiguous`


In [ ]:
model_df = df.copy()
model_df = model_df[model_df[TARGET_COL].notna() & model_df[DATE_COL].notna()].copy()

for col in numeric_features:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

# ZELO POMEMBNO:
# ne uporabljamo pandas StringDtype + pd.NA, ker lahko v nekaterih sklearn verzijah
# povzroči napako "boolean value of NA is ambiguous" med imputacijo / one-hot encodingom.
for col in categorical_base_features + categorical_experimental:
    model_df[col] = model_df[col].astype("object")
    model_df[col] = model_df[col].where(pd.notna(model_df[col]), "__MISSING__")
    model_df[col] = model_df[col].astype(str)

print("model_df shape:", model_df.shape)
display(model_df[[DATE_COL, TARGET_COL, ORIGINAL_TARGET_COL] + numeric_features[:3]].head(3))


NameError: name 'TARGET_COL' is not defined

## 8) Časovni split

### Privzeto
- **train**: `< 2024-01-01`
- **validation**: `2024`
- **test**: `>= 2025-01-01`

Če bi katera množica izpadla prazna, notebook vrže napako.


In [ ]:
VALID_START = pd.Timestamp("2024-01-01")
TEST_START = pd.Timestamp("2025-01-01")

train_mask = model_df[DATE_COL] < VALID_START
valid_mask = (model_df[DATE_COL] >= VALID_START) & (model_df[DATE_COL] < TEST_START)
test_mask = model_df[DATE_COL] >= TEST_START

train_df = model_df[train_mask].copy()
valid_df = model_df[valid_mask].copy()
test_df = model_df[test_mask].copy()

print("train:", train_df.shape)
print("valid:", valid_df.shape)
print("test :", test_df.shape)

if len(train_df) == 0 or len(valid_df) == 0 or len(test_df) == 0:
    raise ValueError("Ena od split množic je prazna. Preveri sale_date range in split datume.")

display(
    pd.DataFrame({
        "split": ["train", "valid", "test"],
        "n_rows": [len(train_df), len(valid_df), len(test_df)],
        "min_date": [train_df[DATE_COL].min(), valid_df[DATE_COL].min(), test_df[DATE_COL].min()],
        "max_date": [train_df[DATE_COL].max(), valid_df[DATE_COL].max(), test_df[DATE_COL].max()],
    })
)


## 9) Helper funkcije za modeliranje

Kategorialni pipeline je narejen robustno:
- najprej imputira manjkajoče na konstanto `__MISSING__`
- nato naredi one-hot encoding


In [ ]:
def build_preprocessor(numeric_cols, categorical_cols):
    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_cols),
            ("cat", categorical_pipe, categorical_cols),
        ],
        remainder="drop",
    )
    return preprocessor

def build_pipeline(numeric_cols, categorical_cols):
    preprocessor = build_preprocessor(numeric_cols, categorical_cols)

    model = RandomForestRegressor(
        n_estimators=400,
        max_depth=20,
        min_samples_leaf=2,
        random_state=SEED,
        n_jobs=-1,
    )

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    return pipe

def evaluate_regression(y_true_log, y_pred_log, y_true_raw):
    y_pred_raw = np.exp(y_pred_log)

    return {
        "mae_log": float(mean_absolute_error(y_true_log, y_pred_log)),
        "rmse_log": rmse(y_true_log, y_pred_log),
        "r2_log": float(r2_score(y_true_log, y_pred_log)),
        "mae_eur_m2": float(mean_absolute_error(y_true_raw, y_pred_raw)),
        "rmse_eur_m2": rmse(y_true_raw, y_pred_raw),
        "r2_eur_m2": float(r2_score(y_true_raw, y_pred_raw)),
    }

def evaluate_constant_baseline(train_target_log, eval_target_log, eval_target_raw):
    baseline_pred_log = np.repeat(np.median(train_target_log), len(eval_target_log))
    return evaluate_regression(eval_target_log, baseline_pred_log, eval_target_raw)

def fit_and_score(model_name, feature_cols, train_df, valid_df):
    X_train = train_df[feature_cols].copy()
    y_train = train_df[TARGET_COL].copy()

    X_valid = valid_df[feature_cols].copy()
    y_valid = valid_df[TARGET_COL].copy()

    pipe = build_pipeline(
        numeric_cols=[c for c in feature_cols if c in numeric_features],
        categorical_cols=[c for c in feature_cols if c in (categorical_base_features + categorical_experimental)],
    )

    pipe.fit(X_train, y_train)

    pred_train = pipe.predict(X_train)
    pred_valid = pipe.predict(X_valid)

    train_metrics = evaluate_regression(y_train, pred_train, train_df[ORIGINAL_TARGET_COL].values)
    valid_metrics = evaluate_regression(y_valid, pred_valid, valid_df[ORIGINAL_TARGET_COL].values)

    row = {"model_name": model_name, "n_features": len(feature_cols), "features": feature_cols}
    row.update({f"train_{k}": v for k, v in train_metrics.items()})
    row.update({f"valid_{k}": v for k, v in valid_metrics.items()})

    return pipe, row

def safe_feature_names(preprocessor, numeric_cols, categorical_cols):
    try:
        return list(preprocessor.get_feature_names_out())
    except Exception:
        names = []
        names.extend([f"num__{c}" for c in numeric_cols])

        cat_encoder = preprocessor.named_transformers_["cat"].named_steps["encoder"]
        cat_names = cat_encoder.get_feature_names_out(categorical_cols)
        names.extend([f"cat__{x}" for x in cat_names])
        return names

def aggregate_importances(feature_names, importances, numeric_cols, categorical_cols):
    rows = []
    for feat_name, imp in zip(feature_names, importances):
        original = None

        for col in numeric_cols:
            if feat_name == f"num__{col}" or feat_name.endswith(f"__{col}"):
                original = col
                break

        if original is None:
            for col in categorical_cols:
                if feat_name.startswith(f"cat__{col}_") or feat_name == f"cat__{col}":
                    original = col
                    break

        if original is None:
            original = feat_name

        rows.append((original, imp))

    agg = pd.DataFrame(rows, columns=["feature_group", "importance"])
    agg = agg.groupby("feature_group", as_index=False)["importance"].sum()
    agg = agg.sort_values("importance", ascending=False).reset_index(drop=True)
    return agg

def transform_to_dense(pipe, X):
    Xt = pipe.named_steps["preprocessor"].transform(X)
    if hasattr(Xt, "toarray"):
        Xt = Xt.toarray()
    return Xt


## 10) Definiraj modela A in B

In [ ]:
FEATURES_A = numeric_features + categorical_base_features
FEATURES_B = numeric_features + categorical_base_features + categorical_experimental

print("FEATURES_A:", FEATURES_A)
print("FEATURES_B:", FEATURES_B)


## 11) Baseline (brez ML)

To je dober sanity check: model mora premagati preprosto konstanto.


In [ ]:
baseline_valid = evaluate_constant_baseline(
    train_df[TARGET_COL].values,
    valid_df[TARGET_COL].values,
    valid_df[ORIGINAL_TARGET_COL].values,
)
baseline_test = evaluate_constant_baseline(
    train_df[TARGET_COL].values,
    test_df[TARGET_COL].values,
    test_df[ORIGINAL_TARGET_COL].values,
)

baseline_df = pd.DataFrame([
    {"model_name": "Baseline_Median", "split": "valid", **baseline_valid},
    {"model_name": "Baseline_Median", "split": "test", **baseline_test},
])
display(baseline_df)


## 12) Treniraj Model A in Model B na train, primerjaj na validation

In [ ]:
pipe_a, row_a = fit_and_score("Model_A_without_trznost", FEATURES_A, train_df, valid_df)
pipe_b, row_b = fit_and_score("Model_B_with_trznost", FEATURES_B, train_df, valid_df)

comparison_df = pd.DataFrame([row_a, row_b]).sort_values("valid_rmse_eur_m2").reset_index(drop=True)
display(comparison_df[[
    "model_name",
    "n_features",
    "train_mae_eur_m2",
    "train_rmse_eur_m2",
    "train_r2_eur_m2",
    "valid_mae_eur_m2",
    "valid_rmse_eur_m2",
    "valid_r2_eur_m2",
    "valid_mae_log",
    "valid_rmse_log",
    "valid_r2_log",
]])


## 13) Izberi boljši model po validation RMSE (`EUR/m²`)

In [ ]:
best_model_name = comparison_df.iloc[0]["model_name"]

if best_model_name == "Model_A_without_trznost":
    best_features = FEATURES_A
else:
    best_features = FEATURES_B

print("Best model by validation RMSE:", best_model_name)
print("Best features:", best_features)


## 14) Retrain najboljšega modela na train + validation, nato eval na test

In [ ]:
trainval_df = pd.concat([train_df, valid_df], ignore_index=True)

best_pipe = build_pipeline(
    numeric_cols=[c for c in best_features if c in numeric_features],
    categorical_cols=[c for c in best_features if c in (categorical_base_features + categorical_experimental)],
)

X_trainval = trainval_df[best_features].copy()
y_trainval = trainval_df[TARGET_COL].copy()

X_test = test_df[best_features].copy()
y_test = test_df[TARGET_COL].copy()

best_pipe.fit(X_trainval, y_trainval)

pred_trainval = best_pipe.predict(X_trainval)
pred_test = best_pipe.predict(X_test)

trainval_metrics = evaluate_regression(y_trainval, pred_trainval, trainval_df[ORIGINAL_TARGET_COL].values)
test_metrics = evaluate_regression(y_test, pred_test, test_df[ORIGINAL_TARGET_COL].values)

final_metrics_df = pd.DataFrame([
    {"model_name": best_model_name, "split": "train+valid", **trainval_metrics},
    {"model_name": best_model_name, "split": "test", **test_metrics},
])
display(final_metrics_df)


## 15) Primerjava z baseline na testu

In [ ]:
test_compare = pd.DataFrame([
    {"model_name": "Baseline_Median", "split": "test", **baseline_test},
    {"model_name": best_model_name, "split": "test", **test_metrics},
])

display(test_compare[[
    "model_name", "split",
    "mae_eur_m2", "rmse_eur_m2", "r2_eur_m2",
    "mae_log", "rmse_log", "r2_log"
]])


## 16) Shrani test napovedi

In [ ]:
test_pred_df = test_df.copy()
test_pred_df["pred_log_price_per_m2"] = pred_test
test_pred_df["pred_price_per_m2"] = np.exp(pred_test)
test_pred_df["abs_error_eur_m2"] = np.abs(test_pred_df["price_per_m2"] - test_pred_df["pred_price_per_m2"])

keep_cols = [
    "sale_date", "municipality", "settlement",
    "price_per_m2", "pred_price_per_m2", "abs_error_eur_m2",
    "price_eur", "area_m2", "build_year", "property_age"
]
keep_cols = [c for c in keep_cols if c in test_pred_df.columns]

display(test_pred_df[keep_cols].head(10))


## 17) Feature importance

Najprej prikažemo:
1. **transformirane featureje** (po one-hot encodingu)
2. **agregirane feature skupine** (bolj uporabno za interpretacijo)


In [ ]:
numeric_used = [c for c in best_features if c in numeric_features]
categorical_used = [c for c in best_features if c in (categorical_base_features + categorical_experimental)]

preprocessor = best_pipe.named_steps["preprocessor"]
rf_model = best_pipe.named_steps["model"]

feature_names = safe_feature_names(preprocessor, numeric_used, categorical_used)
importances = rf_model.feature_importances_

importance_df = pd.DataFrame({
    "feature_name": feature_names,
    "importance": importances,
}).sort_values("importance", ascending=False).reset_index(drop=True)

agg_importance_df = aggregate_importances(feature_names, importances, numeric_used, categorical_used)

display(importance_df.head(20))
display(agg_importance_df.head(20))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
top = agg_importance_df.head(15).sort_values("importance")
ax.barh(top["feature_group"], top["importance"])
ax.set_title("Top aggregated feature importances")
ax.set_xlabel("importance")
plt.show()


## 18) Permutation importance na validation ali test setu

To je počasnejše, ampak pogosto bolj zanesljivo od MDI importance.


In [ ]:
USE_PERMUTATION_IMPORTANCE = True

if USE_PERMUTATION_IMPORTANCE:
    perm_X = X_test
    perm_y = y_test

    perm = permutation_importance(
        best_pipe,
        perm_X,
        perm_y,
        n_repeats=5,
        random_state=SEED,
        n_jobs=-1,
        scoring="neg_root_mean_squared_error",
    )

    perm_df = pd.DataFrame({
        "feature": best_features,
        "perm_importance_mean": perm.importances_mean,
        "perm_importance_std": perm.importances_std,
    }).sort_values("perm_importance_mean", ascending=False).reset_index(drop=True)

    display(perm_df)
else:
    perm_df = pd.DataFrame()
    print("Permutation importance skipped")


## 19) LIME priprava

LIME delamo na **transformiranih featurejih**.

Da ne porabimo preveč pomnilnika:
- explainer zgradimo na vzorcu train+valid transformiranih vrstic
- razložimo nekaj izbranih test primerov


In [ ]:
X_trainval_trans = best_pipe.named_steps["preprocessor"].transform(X_trainval)
X_test_trans = best_pipe.named_steps["preprocessor"].transform(X_test)

if hasattr(X_trainval_trans, "toarray"):
    # vzorec za LIME, da ne dela nepotrebno velikih matric
    lime_sample_size = min(5000, X_trainval_trans.shape[0])
    sample_idx = np.random.RandomState(SEED).choice(X_trainval_trans.shape[0], size=lime_sample_size, replace=False)
    X_lime_background = X_trainval_trans[sample_idx].toarray()
else:
    lime_sample_size = min(5000, X_trainval_trans.shape[0])
    sample_idx = np.random.RandomState(SEED).choice(X_trainval_trans.shape[0], size=lime_sample_size, replace=False)
    X_lime_background = X_trainval_trans[sample_idx]

if hasattr(X_test_trans, "toarray"):
    X_test_dense = X_test_trans.toarray()
else:
    X_test_dense = X_test_trans

transformed_feature_names = safe_feature_names(preprocessor, numeric_used, categorical_used)

print("LIME background shape:", X_lime_background.shape)
print("X_test_dense shape:", X_test_dense.shape)
print("n transformed features:", len(transformed_feature_names))


In [ ]:
def rf_on_transformed(X_array):
    return rf_model.predict(X_array)

lime_explainer = LimeTabularExplainer(
    training_data=X_lime_background,
    feature_names=transformed_feature_names,
    mode="regression",
    random_state=SEED,
    discretize_continuous=True,
)

print("LIME explainer ready")


## 20) Izberi 3 zanimive test primere za razlago

Izberemo:
- nižjo napoved
- median napoved
- višjo napoved


In [ ]:
test_pred_df = test_pred_df.reset_index(drop=True)

quantiles = test_pred_df["pred_price_per_m2"].quantile([0.1, 0.5, 0.9]).values
selected_idx = []

for q in quantiles:
    idx = (test_pred_df["pred_price_per_m2"] - q).abs().idxmin()
    selected_idx.append(int(idx))

selected_idx = list(dict.fromkeys(selected_idx))  # odstrani morebitne duplikate

print("Selected test indices:", selected_idx)
display(test_pred_df.loc[selected_idx, keep_cols])


In [ ]:
lime_rows = []

for idx in selected_idx:
    exp = lime_explainer.explain_instance(
        data_row=X_test_dense[idx],
        predict_fn=rf_on_transformed,
        num_features=10,
    )

    example_meta = test_pred_df.loc[idx, keep_cols].to_dict()
    exp_df = pd.DataFrame(exp.as_list(), columns=["feature", "contribution"])
    exp_df["test_index"] = idx

    lime_rows.append({
        "test_index": idx,
        "meta": example_meta,
        "explanation": exp_df.to_dict(orient="records"),
    })

    print(f"\n=== LIME example idx={idx} ===")
    print(example_meta)
    display(exp_df)

    fig, ax = plt.subplots(figsize=(8, 4))
    plot_df = exp_df.sort_values("contribution")
    ax.barh(plot_df["feature"], plot_df["contribution"])
    ax.set_title(f"LIME explanation for test idx={idx}")
    ax.set_xlabel("contribution to log_price_per_m2")
    plt.show()


## 21) Shrani artefakte

Shrani:
- `comparison_df`
- `final_metrics_df`
- `test_pred_df`
- `importance_df`
- `agg_importance_df`
- `perm_df`
- `best_model_pipeline.joblib`
- `training_metadata.json`
- `lime_examples.json`


In [ ]:
comparison_path = OUT_DIR / "model_comparison_validation.csv"
final_metrics_path = OUT_DIR / "best_model_final_metrics.csv"
test_preds_path = OUT_DIR / "test_predictions.csv"
importance_path = OUT_DIR / "feature_importances_transformed.csv"
agg_importance_path = OUT_DIR / "feature_importances_aggregated.csv"
perm_path = OUT_DIR / "permutation_importance.csv"
model_path = OUT_DIR / "best_model_pipeline.joblib"
metadata_path = OUT_DIR / "training_metadata.json"
lime_path = OUT_DIR / "lime_examples.json"

comparison_df.to_csv(comparison_path, index=False)
final_metrics_df.to_csv(final_metrics_path, index=False)
test_pred_df.to_csv(test_preds_path, index=False)
importance_df.to_csv(importance_path, index=False)
agg_importance_df.to_csv(agg_importance_path, index=False)

if len(perm_df):
    perm_df.to_csv(perm_path, index=False)

joblib.dump(best_pipe, model_path)

metadata = {
    "best_model_name": best_model_name,
    "target_col": TARGET_COL,
    "original_target_col": ORIGINAL_TARGET_COL,
    "date_col": DATE_COL,
    "features_used": best_features,
    "numeric_features_used": numeric_used,
    "categorical_features_used": categorical_used,
    "train_rows": int(len(train_df)),
    "valid_rows": int(len(valid_df)),
    "test_rows": int(len(test_df)),
    "validation_comparison": comparison_df.to_dict(orient="records"),
    "final_metrics": final_metrics_df.to_dict(orient="records"),
    "seed": SEED,
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)

with open(lime_path, "w", encoding="utf-8") as f:
    json.dump(lime_rows, f, ensure_ascii=False, indent=2, default=str)

print("Saved:")
print(" -", comparison_path)
print(" -", final_metrics_path)
print(" -", test_preds_path)
print(" -", importance_path)
print(" -", agg_importance_path)
if len(perm_df):
    print(" -", perm_path)
print(" -", model_path)
print(" -", metadata_path)
print(" -", lime_path)


## 22) Kaj pomeni output

### Če Model B zmaga
Potem `trznost_group` pomaga in jo lahko:
- obdržiš kot eksperimentalni feature
- ali vključiš v “advanced/internal” verzijo modela

### Če Model A zmaga ali je skoraj enak
Potem `trznost_group` verjetno ni potreben za finalni user-facing app.

### Naslednji korak
- priklop najboljšega pipeline-a v Streamlit
- priprava `04_streamlit_slovenia_app.ipynb` ali direktno `app.py`
- integracija LIME explanation view v UI
